In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
# STEPS TO BUILDING OUT A TRANSFORMER NN MODEL: 
# 1. TOKENIZER - Character Level embedding model is simpler - str -> int 
# 2. Select a Model - Bigram Model 
# 3. Optimizer + Training Loop 

# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [7]:
# ML EVAL
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be training data, rest validation
train_data = data[:n]
val_data = data[n:]

In [12]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel (for gpu parallel execution)
block_size = 8 # during training, you cant send the the entire training data at once, u send in periodic batches that forms small relations. 
# the transformer will never receive more than block size of input during inference, so it begins truncating. 

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [14]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)
        if targets == None: 
            loss = None
        else:
            B, T, C = logits.shape # unpack 
            logits = logits.view(B*T, C) # logits.shape = B*T,C
            targets = targets.view(B*T) 
            loss = F.cross_entropy(logits, targets) # since cross_entropy requires B C T 
        return logits, loss


    def generate(self, idx, max_new_tokens): # ALL IT DOES IS GENERATE THE NEXT TOKEN (CHAR) INDEX 
        # idx is (B,T) array of indices in the current context 
        for _ in range(max_new_tokens): 
            logits, loss = self(idx) 
            logits = logits[:, -1, :] # focus on last time step - sets time step to last one in context
            probs = F.softmax(logits, dim=1) 
            idx_next = torch.multinomial(probs, num_samples=1) # B, 1
            idx = torch.cat((idx, idx_next), dim=1) # B, T+1 
        return idx 
        
            
        

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
#print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))
#print(loss)
#print(logits.shape)

# actual_loss = ln(1/65) -- 4.1


In [10]:
# Create a Pytorch optimizer 
# Prev. SGD included iteration, forward pass, Linear Layer, BatchNorm, Backward Pass, Update LR, calculate loss w/lr
# Now - AdamW with lr = 0.001 
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3) 

# training loop
batch_size = 32 
for steps in range(10000): 
    xb, yb = get_batch('train') 
    logits, loss = m(xb, yb) 
    optimizer.zero_grad(set_to_none=True) 
    loss.backward() 
    optimizer.step() 

print(loss.item())

2.382369041442871


In [11]:
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=300)[0].tolist()))



lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecor


In [13]:
torch.manual_seed(1337) 
B, T, C = 4, 8, 2 
x = torch.randn(B,T,C) # batch, time, channel 
# the simplest idea is that you want the T token, to communicate with its previous ones 
# Simple way is to average it, for every batch token, for every t token, calculate the average for all the previos token, and at token T 
x.shape

torch.Size([4, 8, 2])

In [15]:
# VERSION 1 - Brute force with major time complexity 
# we want x[b,t] = mean_{i<=t} x[b,i] 
xbow = torch.zeros((B,T,C))
for b in range(B): 
    for t in range(T): 
        xprev = x[b, :t+1] # t,C where C is all channels for 1 to t (+1 is used bc of range) 
        xbow[b,t] = torch.mean(xprev, 0) # average across the 0th dimension in t,C (vertically down each t, first value in C) 
        # construct a new xbow with shape == x.shape

x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [16]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [20]:
# VERSION 2 - Matrix Method - lower triangle of doom 
# wei == A from brute force method 
# x == B from brute force method 
wei = torch.tril(torch.ones(T,T)) # lower triangle of ones 
wei = wei / wei.sum(1, keepdim=True) # lower triangle of averages across length of dimension
xbow2 = wei @ x # (B,T,T) @ (B,T,C) ---> Batch matrix multiplication, secretly adds B in the original T,T in wei and does it for all batches
#torch.allclose(xbow, xbow2, atol=1e-6) similarity check

xbow2


tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [47]:
# VERSION 3 - Apply Softmax
tril = torch.tril(torch.ones(T,T)) # lower traingle of 1s 
wei = torch.zeros(T,T) # 8,8 of all zeros
wei = wei.masked_fill(tril==0, float('-inf')) # take all 1s from tril and make it 0., and turn all 0s into -inf
wei = F.softmax(wei, dim=-1) # since softmax is also a normalization operation so its traingle of infs to a mean 8,8 of averages
# in theory, softmax squares by its base and divides each number by sum
wei



tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [37]:
# VERSION 4 - IMPLEMENT SELF ATTENTION (SINGULAR HEAD) 
torch.manual_seed(1337) 
B,T,C = 4,8,32 
x = torch.randn(B,T,C) 
# so far each token contains information from its embedding and position

# let's see a single head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # (B, T, 16)
q = query(x) # (B, T, 16)
# So far the two vectors have been created independantly 

# Now you want them to communicate (query @ all_keys)
wei = q @ k.transpose(-2, -1) ** head_size**-0.5 
# (B,T,16)@(B,16,T) --> (B, T, T)
# diffuse the variance of wei otherwise it will be too peaky and biased during initialization

tril = torch.tril(torch.ones(T, T)) # represents the rule that tokens can only talk to tokens before them. 
#wei = torch.zeros((T,T)) # 8,8
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)=

v = value(x) # each token has some value association
out = wei @ v # aggregate on the values
# now each tokens representation contains value from previous tokens (when u multiplied attention scores to values)

# out is now an attention matrix 
out.shape

torch.Size([4, 8, 16])

In [39]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [ ]:
# NOTES


# 1. Attention is a communication mechanism 
# 2. There is no notion of space. Attention simply acts over vectors.This is why we encode positon tokens
# 3. Each example in batch dimension are independant nodes and never talk to each other 
# 4. An encoder block doesnt allow masking triangle, allowing all nodes to talk to each other. (Pretty good for sentiment analysis) 
# Our implementation has this autoregressive format where you mask future nodes. This is called decoding block 
# 5. WHAT IS SELF ATTENTION VS CROSS ATTENTION: self attention means that all the query, value, key come from same source, x. 
# 6. Why is the paper doing skaled attention (dividing my sqrt of head size): Because when query and key are unit gausian inputs (0 mean unit variance)
# if you dot product it naively, the wei variance will be == headsize, but if you normalize it, the variance is closer to gaussian. 